# Phishing Website Detection Using Machine Learning
## CSAI412 Machine Learning Group Project

---

**Dataset:** UCI Phishing Websites Dataset (Mohammad, Thabtah & McCluskey)  
**Samples:** 11,055 websites  
**Features:** 30 URL-based and website-based characteristics  
**Target:** `Result` (-1 = Phishing, 1 = Legitimate)  
**Feature encoding:** All features are ternary {-1, 0, 1} representing Suspicious / Neutral / Legitimate  

This notebook performs a complete machine learning pipeline:
1. Data loading and exploration
2. Comprehensive Exploratory Data Analysis (EDA)
3. Training and evaluation of 7 models (Logistic Regression, KNN, SVM Linear, SVM RBF, Decision Tree, MLP, K-Means)
4. Final comparison with statistical significance testing

**Reference:** R.M. Mohammad, F. Thabtah, L. McCluskey. *Phishing Websites Features.* University of Huddersfield.

---
## 1. Introduction & Setup

We begin by importing all necessary libraries and configuring the plotting style.

In [ ]:
# Standard libraries
import os
import warnings
import numpy as np
import pandas as pd

# Visualization
import matplotlib
import matplotlib.pyplot as plt
import matplotlib.gridspec as gridspec
import seaborn as sns

# Scikit-learn: preprocessing
from sklearn.model_selection import train_test_split, GridSearchCV, cross_val_score
from sklearn.preprocessing import StandardScaler, label_binarize

# Scikit-learn: models
from sklearn.linear_model import LogisticRegression
from sklearn.neighbors import KNeighborsClassifier
from sklearn.svm import SVC
from sklearn.tree import DecisionTreeClassifier, plot_tree, export_text
from sklearn.neural_network import MLPClassifier
from sklearn.cluster import KMeans
from sklearn.decomposition import PCA

# Scikit-learn: metrics
from sklearn.metrics import (
    accuracy_score, precision_score, recall_score, f1_score,
    classification_report, confusion_matrix, roc_curve, auc, roc_auc_score
)

# Feature selection & statistical tests
from sklearn.feature_selection import mutual_info_classif
from sklearn.ensemble import RandomForestClassifier
from scipy import stats

# Inline plotting
%matplotlib inline

# Suppress warnings for cleaner output
warnings.filterwarnings('ignore')

# Style configuration
sns.set_theme(style='whitegrid', palette='muted', font_scale=1.1)
plt.rcParams.update({
    'figure.dpi': 120,
    'figure.facecolor': 'white',
    'axes.facecolor': 'white',
    'font.size': 11,
})

# Color palettes
CLASS_PALETTE = {'Phishing': '#e74c3c', 'Legitimate': '#2ecc71'}
CLASS_COLORS = ['#e74c3c', '#2ecc71']  # index 0=Phishing, 1=Legitimate
VALUE_PALETTE = {-1: '#e74c3c', 0: '#f39c12', 1: '#2ecc71'}
CLASS_NAMES = ['Phishing', 'Legitimate']

# Random seed for reproducibility
RANDOM_STATE = 42
np.random.seed(RANDOM_STATE)

print('All imports successful.')
print(f'NumPy: {np.__version__}')
print(f'Pandas: {pd.__version__}')
print(f'Matplotlib: {matplotlib.__version__}')
print(f'Seaborn: {sns.__version__}')

---
## 2. Data Loading

We load the phishing dataset from `data/phishing.csv`. The CSV has a trailing comma that creates a junk column (`Unnamed: 31`) which we drop.

In [ ]:
# Load dataset
df = pd.read_csv('data/phishing.csv')

# Drop the junk column caused by the trailing comma in the CSV
if 'Unnamed: 31' in df.columns:
    df = df.drop(columns=['Unnamed: 31'])
    print('Dropped junk column: Unnamed: 31')

print(f'\nDataset shape: {df.shape[0]} samples x {df.shape[1]} columns')
print(f'Memory usage: {df.memory_usage(deep=True).sum() / 1024:.1f} KB')
print(f'\nTarget column: Result')
print(f'Target values: {sorted(df["Result"].unique())} (-1=Phishing, 1=Legitimate)')
print(f'\nFirst 5 rows:')
df.head()

In [ ]:
# Dataset info
print('Dataset Info:')
print(f'  Rows:    {df.shape[0]}')
print(f'  Columns: {df.shape[1]}')
print(f'  Missing values: {df.isna().sum().sum()}')
print(f'  Duplicates: {df.duplicated().sum()}')
print(f'\nData types:')
print(df.dtypes.value_counts())
print(f'\nBasic statistics:')
df.describe()

---
## 3. EDA Section 1: Class Distribution

We first examine the distribution of the target variable `Result`. A balanced dataset simplifies model training as we do not need resampling techniques.

In [ ]:
# Map target values for display
class_map = {-1: 'Phishing', 1: 'Legitimate'}
class_labels = df['Result'].map(class_map)
class_dist = class_labels.value_counts()
class_pct = class_labels.value_counts(normalize=True) * 100

print('Class Distribution:')
for cls in ['Phishing', 'Legitimate']:
    print(f'  {cls:<12} {class_dist[cls]:>6} samples  ({class_pct[cls]:.2f}%)')

imbalance_ratio = class_dist.max() / class_dist.min()
print(f'\nImbalance ratio: {imbalance_ratio:.2f}:1')
print(f'Assessment: {"Near-perfect balance" if imbalance_ratio < 1.5 else "Moderate imbalance"}')

# Visualization
fig, axes = plt.subplots(1, 2, figsize=(14, 5))

# Bar plot
colors = [CLASS_PALETTE['Phishing'], CLASS_PALETTE['Legitimate']]
bar_data = [class_dist.get('Phishing', 0), class_dist.get('Legitimate', 0)]
bars = axes[0].bar(['Phishing', 'Legitimate'], bar_data, color=colors, edgecolor='black', linewidth=0.5)
axes[0].set_xlabel('Class')
axes[0].set_ylabel('Count')
axes[0].set_title('Class Distribution', fontweight='bold')
for i, cnt in enumerate(bar_data):
    axes[0].text(i, cnt + 50, str(cnt), ha='center', va='bottom', fontsize=12, fontweight='bold')

# Pie chart
axes[1].pie(bar_data, labels=['Phishing', 'Legitimate'],
            colors=colors, autopct='%1.1f%%', startangle=90, pctdistance=0.85,
            textprops={'fontsize': 12}, wedgeprops={'edgecolor': 'white', 'linewidth': 1.5})
axes[1].set_title('Class Proportions', fontweight='bold')

plt.suptitle('Phishing Website Dataset -- Class Distribution Analysis',
             fontsize=14, fontweight='bold', y=1.02)
plt.tight_layout()
plt.show()

**Findings:** The dataset is nearly balanced with approximately 44% Phishing and 56% Legitimate websites (ratio ~1.26:1). This means we do not need to apply oversampling (SMOTE) or undersampling techniques -- standard accuracy is a valid evaluation metric.

---
## 4. EDA Section 2: Feature Analysis

All 30 features are encoded as ternary values {-1, 0, 1}:
- **-1** = Suspicious / Phishing indicator
- **0** = Neutral / Uncertain
- **1** = Legitimate indicator

We examine the value counts for each feature and compare distributions between Phishing and Legitimate classes.

In [ ]:
feature_cols = [col for col in df.columns if col != 'Result']
n_features = len(feature_cols)

# Value counts per feature
print(f'Value Counts per Feature (all features are in {{-1, 0, 1}}):')
print(f'{"Feature":<35} {"-1 (Suspicious)":>18} {"0 (Neutral)":>14} {"1 (Legitimate)":>17}')
print('-' * 87)
for col in feature_cols:
    vc = df[col].value_counts().sort_index()
    v_neg = vc.get(-1, 0)
    v_zero = vc.get(0, 0)
    v_pos = vc.get(1, 0)
    print(f'{col:<35} {v_neg:>8} ({v_neg/len(df)*100:5.1f}%) {v_zero:>6} ({v_zero/len(df)*100:5.1f}%) {v_pos:>8} ({v_pos/len(df)*100:5.1f}%)')

# Binary vs ternary features
binary_feats = [c for c in feature_cols if df[c].nunique() == 2]
ternary_feats = [c for c in feature_cols if df[c].nunique() == 3]
print(f'\nBinary features (only -1 and 1): {len(binary_feats)}')
print(f'Ternary features (-1, 0, and 1): {len(ternary_feats)}')

In [ ]:
# Feature value distributions -- overall
n_cols_grid = 5
n_rows_grid = (n_features + n_cols_grid - 1) // n_cols_grid

fig, axes = plt.subplots(n_rows_grid, n_cols_grid, figsize=(22, n_rows_grid * 3.5))
axes_flat = axes.flatten()

for i, col in enumerate(feature_cols):
    ax = axes_flat[i]
    vc = df[col].value_counts().sort_index()
    bar_colors = [VALUE_PALETTE.get(v, '#999') for v in vc.index]
    ax.bar([str(v) for v in vc.index], vc.values, color=bar_colors,
           edgecolor='black', linewidth=0.3)
    ax.set_title(col, fontsize=9, fontweight='bold')
    ax.set_ylabel('Count')
    for j, (val, cnt) in enumerate(vc.items()):
        ax.text(j, cnt + 30, str(cnt), ha='center', va='bottom', fontsize=7)

for j in range(i + 1, len(axes_flat)):
    axes_flat[j].set_visible(False)

plt.suptitle('Feature Value Distributions (Red=-1 Suspicious, Orange=0 Neutral, Green=1 Legitimate)',
             fontsize=13, fontweight='bold', y=1.01)
plt.tight_layout()
plt.show()

In [ ]:
# Feature distributions by class (Phishing vs Legitimate)
df_labeled = df.copy()
df_labeled['Class'] = df_labeled['Result'].map(class_map)

fig, axes = plt.subplots(n_rows_grid, n_cols_grid, figsize=(22, n_rows_grid * 3.5))
axes_flat = axes.flatten()

for i, col in enumerate(feature_cols):
    ax = axes_flat[i]
    for cls_name, cls_color in [('Phishing', '#e74c3c'), ('Legitimate', '#2ecc71')]:
        subset = df_labeled[df_labeled['Class'] == cls_name][col]
        vc = subset.value_counts(normalize=True).sort_index()
        x_positions = np.array([v for v in vc.index])
        offset = -0.15 if cls_name == 'Phishing' else 0.15
        ax.bar(x_positions + offset, vc.values, width=0.3, color=cls_color,
               alpha=0.8, edgecolor='black', linewidth=0.2, label=cls_name)
    ax.set_title(col, fontsize=9, fontweight='bold')
    ax.set_ylabel('Proportion')
    ax.set_xticks([-1, 0, 1])
    if i == 0:
        ax.legend(fontsize=7)

for j in range(i + 1, len(axes_flat)):
    axes_flat[j].set_visible(False)

plt.suptitle('Feature Value Proportions by Class (Phishing vs Legitimate)',
             fontsize=13, fontweight='bold', y=1.01)
plt.tight_layout()
plt.show()

In [ ]:
# Stacked bar chart: value proportions per class for each feature
fig, axes = plt.subplots(n_rows_grid, n_cols_grid, figsize=(22, n_rows_grid * 4))
axes_flat = axes.flatten()

for i, col in enumerate(feature_cols):
    ax = axes_flat[i]
    ct = pd.crosstab(df_labeled['Class'], df_labeled[col], normalize='index')
    ct = ct.reindex(columns=[-1, 0, 1], fill_value=0)
    ct.plot(kind='bar', ax=ax,
            color=[VALUE_PALETTE[-1], VALUE_PALETTE[0], VALUE_PALETTE[1]],
            edgecolor='black', linewidth=0.3, width=0.7)
    ax.set_title(col, fontsize=9, fontweight='bold')
    ax.set_ylabel('Proportion')
    ax.set_xticklabels(['Phishing', 'Legitimate'], rotation=0, fontsize=8)
    ax.legend(title='Value', fontsize=6, title_fontsize=7)
    ax.set_ylim(0, 1.05)

for j in range(i + 1, len(axes_flat)):
    axes_flat[j].set_visible(False)

plt.suptitle('Feature Value Proportions by Class (Stacked: -1=Red, 0=Orange, 1=Green)',
             fontsize=13, fontweight='bold', y=1.01)
plt.tight_layout()
plt.show()

**Findings:** Features like `SSLfinal_State`, `URL_of_Anchor`, `having_Sub_Domain`, and `web_traffic` show strong discriminating power -- the value distributions differ substantially between Phishing and Legitimate classes. Features like `Redirect` and `RightClick` show less class separation.

---
## 5. EDA Section 3: Correlation Analysis

We compute Pearson correlations between all features and with the target. Strong correlations with `Result` indicate features that are most useful for classification. Feature-feature correlations may indicate multicollinearity.

In [ ]:
# Full correlation heatmap
corr_features = df[feature_cols].corr()

fig, ax = plt.subplots(figsize=(18, 15))
mask = np.triu(np.ones_like(corr_features, dtype=bool), k=1)
sns.heatmap(corr_features, mask=mask, annot=True, fmt='.2f', cmap='RdBu_r',
            center=0, vmin=-1, vmax=1, square=True,
            linewidths=0.5, linecolor='white',
            cbar_kws={'shrink': 0.8, 'label': 'Pearson Correlation'},
            ax=ax, annot_kws={'size': 7})
ax.set_title('Correlation Heatmap -- Phishing Website Features',
             fontsize=14, fontweight='bold', pad=20)
ax.tick_params(axis='x', rotation=45, labelsize=8)
ax.tick_params(axis='y', labelsize=8)
plt.tight_layout()
plt.show()

In [ ]:
# Correlation with target
corr_with_target = df[feature_cols + ['Result']].corr()['Result'].drop('Result').sort_values(key=abs, ascending=False)

print('Feature Correlation with Target (Result):')
print(f'{"Feature":<35} {"Correlation":<12} {"Strength"}')
print('-' * 60)
for feat, val in corr_with_target.items():
    strength = 'STRONG' if abs(val) > 0.3 else ('MODERATE' if abs(val) > 0.15 else 'WEAK')
    print(f'{feat:<35} {val:>+.4f}       {strength}')

# Bar chart of correlation with target
fig, ax = plt.subplots(figsize=(10, 10))
corr_sorted = corr_with_target.sort_values()
colors = ['#2ecc71' if v > 0 else '#e74c3c' for v in corr_sorted.values]
bars = ax.barh(corr_sorted.index, corr_sorted.values, color=colors, edgecolor='black', linewidth=0.5)
ax.set_xlabel('Pearson Correlation Coefficient')
ax.set_title('Feature Correlation with Phishing Result (Target)',
             fontsize=13, fontweight='bold')
ax.axvline(x=0, color='black', linewidth=0.8)
for bar, val in zip(bars, corr_sorted.values):
    ax.text(val + (0.01 if val > 0 else -0.01), bar.get_y() + bar.get_height()/2,
            f'{val:+.3f}', ha='left' if val > 0 else 'right', va='center', fontsize=8)
ax.tick_params(axis='y', labelsize=9)
plt.tight_layout()
plt.show()

In [ ]:
# Feature-feature correlations (multicollinearity check)
print('Top Feature-Feature Correlations (|r| > 0.3, potential multicollinearity):')
pairs = []
for i, c1 in enumerate(feature_cols):
    for c2 in feature_cols[i+1:]:
        r = corr_features.loc[c1, c2]
        if abs(r) > 0.3:
            pairs.append((c1, c2, r))
pairs.sort(key=lambda x: abs(x[2]), reverse=True)

if pairs:
    for c1, c2, r in pairs:
        print(f'  {c1:<30} <-> {c2:<30} r={r:+.4f}')
else:
    print('  (none found -- features are largely independent)')

**Findings:** `SSLfinal_State`, `URL_of_Anchor`, `having_Sub_Domain`, and `web_traffic` show the strongest correlations with the target. The low inter-feature correlations indicate minimal multicollinearity, meaning each feature provides relatively independent information.

---
## 6. EDA Section 4: Feature Importance

We use two model-based methods to assess feature importance:
1. **Mutual Information** -- measures the mutual dependence between each feature and the target (non-linear, information-theoretic)
2. **Random Forest Importance** -- uses Gini impurity reduction in an ensemble of decision trees

In [ ]:
X_raw = df[feature_cols].values
y_raw = np.where(df['Result'].values == -1, 0, 1)  # Remap: -1 -> 0 (Phishing), 1 -> 1 (Legitimate)

# Mutual Information
mi_scores = mutual_info_classif(X_raw, y_raw, random_state=RANDOM_STATE)
mi_df = pd.DataFrame({'Feature': feature_cols, 'MI Score': mi_scores}).sort_values('MI Score', ascending=False)

# Random Forest importance
rf = RandomForestClassifier(n_estimators=100, random_state=RANDOM_STATE, n_jobs=-1)
rf.fit(X_raw, y_raw)
rf_imp = rf.feature_importances_
rf_df = pd.DataFrame({'Feature': feature_cols, 'RF Importance': rf_imp}).sort_values('RF Importance', ascending=False)

# Combined ranking
combined = mi_df.copy()
combined['MI Rank'] = range(1, len(combined) + 1)
combined = combined.merge(rf_df, on='Feature')
combined = combined.sort_values('RF Importance', ascending=False)
combined['RF Rank'] = range(1, len(combined) + 1)
combined['Avg Rank'] = (combined['MI Rank'] + combined['RF Rank']) / 2
combined = combined.sort_values('Avg Rank')

print('Feature Importance Ranking:')
print(f'{"Feature":<35} {"MI Score":>10} {"MI Rank":>8} {"RF Imp":>10} {"RF Rank":>8} {"Avg Rank":>9}')
print('-' * 84)
for _, row in combined.iterrows():
    print(f'{row["Feature"]:<35} {row["MI Score"]:>10.4f} {int(row["MI Rank"]):>8} '
          f'{row["RF Importance"]:>10.4f} {int(row["RF Rank"]):>8} {row["Avg Rank"]:>9.1f}')

In [ ]:
# Feature importance visualization
fig, axes = plt.subplots(1, 2, figsize=(18, 9))

# Mutual Information
mi_sorted = mi_df.sort_values('MI Score', ascending=True)
axes[0].barh(mi_sorted['Feature'], mi_sorted['MI Score'], color='#3498db', edgecolor='black', linewidth=0.5)
axes[0].set_xlabel('Mutual Information Score')
axes[0].set_title('Mutual Information with Phishing Result', fontsize=12, fontweight='bold')
axes[0].tick_params(axis='y', labelsize=9)

# Random Forest importance
rf_sorted = rf_df.sort_values('RF Importance', ascending=True)
axes[1].barh(rf_sorted['Feature'], rf_sorted['RF Importance'], color='#e67e22', edgecolor='black', linewidth=0.5)
axes[1].set_xlabel('Random Forest Feature Importance')
axes[1].set_title('Random Forest Importance', fontsize=12, fontweight='bold')
axes[1].tick_params(axis='y', labelsize=9)

plt.suptitle('Feature Importance Analysis -- Phishing Website Detection',
             fontsize=14, fontweight='bold', y=1.02)
plt.tight_layout()
plt.show()

**Findings:** Both methods consistently rank `SSLfinal_State`, `URL_of_Anchor`, `web_traffic`, and `having_Sub_Domain` as the most important features. This aligns with intuition -- SSL certificate validity, anchor URL destinations, and web traffic rank are strong indicators of website legitimacy.

---
## 7. EDA Section 5: Statistical Tests

We perform the Chi-Squared test of independence for each feature against the target class. Since all features and the target are categorical, this test checks whether each feature's distribution is statistically different between Phishing and Legitimate websites.

In [ ]:
# Chi-squared test of independence
print('Chi-Squared Test of Independence (Feature vs Class):')
print(f'{"Feature":<35} {"Chi2":>10} {"p-value":>12} {"DoF":>5} {"Significant?"}')
print('-' * 75)

chi2_results = []
for col in feature_cols:
    ct = pd.crosstab(df['Result'], df[col])
    chi2, p_val, dof, expected = stats.chi2_contingency(ct)
    sig = '*** YES' if p_val < 0.001 else ('** YES' if p_val < 0.01 else ('* YES' if p_val < 0.05 else 'NO'))
    print(f'{col:<35} {chi2:>10.2f} {p_val:>12.2e} {dof:>5} {sig}')
    chi2_results.append({'Feature': col, 'Chi2': chi2, 'p_value': p_val, 'Significant': p_val < 0.05})

chi2_df = pd.DataFrame(chi2_results)
sig_count = chi2_df['Significant'].sum()
print(f'\nSignificant features (p < 0.05): {sig_count}/{len(feature_cols)}')

# Visualize chi-squared statistics
chi2_sorted = chi2_df.sort_values('Chi2', ascending=True)

fig, ax = plt.subplots(figsize=(10, 10))
colors_chi = ['#2ecc71' if s else '#e74c3c' for s in chi2_sorted['Significant']]
ax.barh(chi2_sorted['Feature'], chi2_sorted['Chi2'], color=colors_chi, edgecolor='black', linewidth=0.5)
ax.set_xlabel('Chi-Squared Statistic')
ax.set_title('Chi-Squared Test Statistics (Green = Significant at p < 0.05)',
             fontsize=13, fontweight='bold')
ax.tick_params(axis='y', labelsize=9)
plt.tight_layout()
plt.show()

**Findings:** The vast majority of features show statistically significant association with the target class (p < 0.001), confirming that these URL and website characteristics are indeed informative for distinguishing phishing from legitimate websites.

---
## 8. Data Preprocessing

We prepare the data for model training:
1. Separate features (X) and target (y)
2. Remap target: -1 (Phishing) -> 0, 1 (Legitimate) -> 1
3. Apply `StandardScaler` for zero mean and unit variance
4. Stratified train/test split (80/20)

In [ ]:
# Separate features and target
X = df[feature_cols].values.astype(np.float64)
y = np.where(df['Result'].values == -1, 0, 1)  # 0=Phishing, 1=Legitimate

print(f'Features shape: {X.shape}')
print(f'Target shape: {y.shape}')
print(f'Target distribution: Phishing(0)={np.sum(y==0)}, Legitimate(1)={np.sum(y==1)}')

# Apply StandardScaler
scaler = StandardScaler()
X_scaled = scaler.fit_transform(X)
print(f'\nAfter scaling:')
print(f'  Mean range: [{X_scaled.mean(axis=0).min():.6f}, {X_scaled.mean(axis=0).max():.6f}] (should be ~0)')
print(f'  Std range:  [{X_scaled.std(axis=0).min():.6f}, {X_scaled.std(axis=0).max():.6f}] (should be ~1)')

# Stratified train/test split
X_train, X_test, y_train, y_test = train_test_split(
    X_scaled, y, test_size=0.2, random_state=RANDOM_STATE, stratify=y
)

print(f'\nTrain/Test Split:')
print(f'  Training set:  {X_train.shape[0]} samples (80%)')
print(f'  Test set:      {X_test.shape[0]} samples (20%)')
print(f'\nClass distribution verification:')
for label, name in [(0, 'Phishing'), (1, 'Legitimate')]:
    train_pct = np.mean(y_train == label) * 100
    test_pct = np.mean(y_test == label) * 100
    print(f'  {name:<12}  Train: {train_pct:.1f}%  Test: {test_pct:.1f}%')

In [ ]:
# Visualize the effect of scaling
fig, axes = plt.subplots(1, 2, figsize=(16, 5))

# Before scaling: histogram of a few features
sample_feats = [0, 5, 10, 15, 20]
for idx in sample_feats:
    axes[0].hist(X[:, idx], bins=5, alpha=0.5, label=feature_cols[idx], edgecolor='black')
axes[0].set_title('Before StandardScaler (5 sample features)', fontweight='bold')
axes[0].set_xlabel('Feature Value')
axes[0].set_ylabel('Count')
axes[0].legend(fontsize=7)

# After scaling
for idx in sample_feats:
    axes[1].hist(X_scaled[:, idx], bins=20, alpha=0.5, label=feature_cols[idx], edgecolor='black')
axes[1].set_title('After StandardScaler (same features)', fontweight='bold')
axes[1].set_xlabel('Scaled Feature Value')
axes[1].set_ylabel('Count')
axes[1].legend(fontsize=7)

plt.suptitle('Effect of StandardScaler on Feature Distributions', fontsize=14, fontweight='bold', y=1.02)
plt.tight_layout()
plt.show()

---
## 9. Model 1: Logistic Regression (Baseline)

Logistic Regression is our **baseline** classifier. It uses a linear decision boundary (hyperplane) and the sigmoid function to produce class probabilities. All subsequent models will be compared against this baseline.

**Why Logistic Regression?**
- Simple, fast, and interpretable
- Feature coefficients reveal which URL/website properties are most indicative of phishing
- If it performs well, the phishing signal is largely linearly separable

In [ ]:
# Hyperparameter tuning with GridSearchCV
print('Logistic Regression -- Hyperparameter Tuning (GridSearchCV, 5-fold CV)')
print('=' * 65)

lr_param_grid = {
    'C': [0.001, 0.01, 0.1, 1, 10, 100],
    'solver': ['lbfgs', 'newton-cg'],
}

lr = LogisticRegression(max_iter=1000, random_state=RANDOM_STATE)
lr_grid = GridSearchCV(
    estimator=lr,
    param_grid=lr_param_grid,
    cv=5,
    scoring='accuracy',
    n_jobs=-1,
    return_train_score=True,
)
lr_grid.fit(X_train, y_train)

print(f'\nBest parameters: {lr_grid.best_params_}')
print(f'Best CV accuracy: {lr_grid.best_score_:.4f}')

# Show results
lr_results = pd.DataFrame(lr_grid.cv_results_).sort_values('rank_test_score')
print(f'\n{"Rank":<6} {"C":<10} {"Solver":<12} {"Mean CV Acc":<14} {"Std":<10}')
print('-' * 52)
for _, row in lr_results.head(6).iterrows():
    print(f'{int(row["rank_test_score"]):<6} {row["param_C"]:<10} {row["param_solver"]:<12} '
          f'{row["mean_test_score"]:<14.4f} {row["std_test_score"]:<10.4f}')

lr_best = lr_grid.best_estimator_

In [ ]:
# Evaluation on test set
lr_pred = lr_best.predict(X_test)
lr_acc = accuracy_score(y_test, lr_pred)
lr_prec = precision_score(y_test, lr_pred, average='weighted')
lr_rec = recall_score(y_test, lr_pred, average='weighted')
lr_f1 = f1_score(y_test, lr_pred, average='weighted')

print('Logistic Regression -- Test Set Results')
print('=' * 45)
print(f'Accuracy:  {lr_acc:.4f}')
print(f'Precision: {lr_prec:.4f}')
print(f'Recall:    {lr_rec:.4f}')
print(f'F1-Score:  {lr_f1:.4f}')
print(f'\n{classification_report(y_test, lr_pred, target_names=CLASS_NAMES)}')

# Confusion matrix
lr_cm = confusion_matrix(y_test, lr_pred)

fig, axes = plt.subplots(1, 2, figsize=(16, 5.5))

# Confusion matrix heatmap
sns.heatmap(lr_cm, annot=True, fmt='d', cmap='Blues',
            xticklabels=CLASS_NAMES, yticklabels=CLASS_NAMES,
            ax=axes[0], linewidths=0.5, linecolor='gray')
axes[0].set_xlabel('Predicted')
axes[0].set_ylabel('Actual')
axes[0].set_title(f'Logistic Regression Confusion Matrix\nAccuracy: {lr_acc:.4f}',
                  fontweight='bold')

# ROC curve
lr_prob = lr_best.predict_proba(X_test)[:, 1]
lr_fpr, lr_tpr, _ = roc_curve(y_test, lr_prob)
lr_auc = auc(lr_fpr, lr_tpr)

axes[1].plot(lr_fpr, lr_tpr, color='darkorange', lw=2, label=f'ROC curve (AUC = {lr_auc:.3f})')
axes[1].plot([0, 1], [0, 1], 'k--', lw=1, label='Random (AUC = 0.500)')
axes[1].set_xlim([-0.02, 1.02])
axes[1].set_ylim([-0.02, 1.02])
axes[1].set_xlabel('False Positive Rate')
axes[1].set_ylabel('True Positive Rate')
axes[1].set_title(f'Logistic Regression -- ROC Curve\nAUC = {lr_auc:.3f}', fontweight='bold')
axes[1].legend(loc='lower right')
axes[1].grid(True, alpha=0.3)

plt.tight_layout()
plt.show()

In [ ]:
# Feature coefficients (interpretability)
lr_coefs = lr_best.coef_[0]
coef_df = pd.DataFrame({
    'Feature': feature_cols,
    'Coefficient': lr_coefs,
    '|Coefficient|': np.abs(lr_coefs),
}).sort_values('|Coefficient|', ascending=True)

fig, ax = plt.subplots(figsize=(10, 9))
colors = ['#d32f2f' if c < 0 else '#388e3c' for c in coef_df['Coefficient']]
ax.barh(coef_df['Feature'], coef_df['Coefficient'], color=colors, edgecolor='gray')
ax.set_xlabel('Coefficient Value')
ax.set_title('Logistic Regression Feature Coefficients\n(Negative = Phishing indicator, Positive = Legitimate indicator)',
             fontsize=13, fontweight='bold')
ax.axvline(x=0, color='black', linewidth=0.8)
ax.grid(True, axis='x', alpha=0.3)
plt.tight_layout()
plt.show()

---
## 10. Model 2: K-Nearest Neighbours (KNN)

KNN is a non-parametric, instance-based learner that classifies a new sample by majority vote among its k nearest neighbours in feature space.

**Why KNN?**
- No assumptions about data distribution
- Naturally captures non-linear decision boundaries
- With 30 ternary features, distance metrics can be meaningful

In [ ]:
# Hyperparameter tuning
print('KNN -- Hyperparameter Tuning (GridSearchCV, 5-fold CV)')
print('=' * 55)

knn_param_grid = {
    'n_neighbors': [1, 3, 5, 7, 9, 11, 15, 21],
    'weights': ['uniform', 'distance'],
    'metric': ['euclidean', 'manhattan'],
}

knn = KNeighborsClassifier()
knn_grid = GridSearchCV(
    estimator=knn,
    param_grid=knn_param_grid,
    cv=5,
    scoring='accuracy',
    n_jobs=-1,
    return_train_score=True,
)
knn_grid.fit(X_train, y_train)

print(f'\nBest parameters: {knn_grid.best_params_}')
print(f'Best CV accuracy: {knn_grid.best_score_:.4f}')

knn_best = knn_grid.best_estimator_

In [ ]:
# k vs Accuracy curve
k_values = [1, 3, 5, 7, 9, 11, 13, 15, 17, 19, 21, 25, 31]
train_accs = []
test_accs = []

best_weight = knn_grid.best_params_.get('weights', 'distance')
best_metric = knn_grid.best_params_.get('metric', 'euclidean')

for k in k_values:
    knn_temp = KNeighborsClassifier(n_neighbors=k, weights=best_weight, metric=best_metric)
    knn_temp.fit(X_train, y_train)
    train_accs.append(accuracy_score(y_train, knn_temp.predict(X_train)))
    test_accs.append(accuracy_score(y_test, knn_temp.predict(X_test)))

fig, ax = plt.subplots(figsize=(10, 6))
ax.plot(k_values, train_accs, 'o-', label='Training Accuracy', color='#2196F3', linewidth=2, markersize=7)
ax.plot(k_values, test_accs, 's-', label='Test Accuracy', color='#FF5722', linewidth=2, markersize=7)

best_k = knn_grid.best_params_.get('n_neighbors', 5)
if best_k in k_values:
    idx = k_values.index(best_k)
    ax.axvline(x=best_k, color='green', linestyle='--', alpha=0.7, label=f'Best k={best_k}')
    ax.scatter([best_k], [test_accs[idx]], color='green', s=150, zorder=5, edgecolors='black', linewidth=2)

ax.set_xlabel('Number of Neighbors (k)', fontsize=13)
ax.set_ylabel('Accuracy', fontsize=13)
ax.set_title('KNN: k vs Accuracy (Phishing Detection)', fontsize=15, fontweight='bold')
ax.legend(fontsize=11)
ax.set_xticks(k_values)
ax.grid(True, alpha=0.3)
plt.tight_layout()
plt.show()

In [ ]:
# Evaluation on test set
knn_pred = knn_best.predict(X_test)
knn_acc = accuracy_score(y_test, knn_pred)
knn_prec = precision_score(y_test, knn_pred, average='weighted')
knn_rec = recall_score(y_test, knn_pred, average='weighted')
knn_f1 = f1_score(y_test, knn_pred, average='weighted')

print('KNN -- Test Set Results')
print('=' * 45)
print(f'Accuracy:  {knn_acc:.4f}')
print(f'Precision: {knn_prec:.4f}')
print(f'Recall:    {knn_rec:.4f}')
print(f'F1-Score:  {knn_f1:.4f}')
print(f'\n{classification_report(y_test, knn_pred, target_names=CLASS_NAMES)}')

# Confusion matrix
knn_cm = confusion_matrix(y_test, knn_pred)

fig, ax = plt.subplots(figsize=(7, 5.5))
sns.heatmap(knn_cm, annot=True, fmt='d', cmap='Blues',
            xticklabels=CLASS_NAMES, yticklabels=CLASS_NAMES,
            ax=ax, linewidths=0.5)
ax.set_xlabel('Predicted')
ax.set_ylabel('Actual')
ax.set_title(f'KNN Confusion Matrix (k={knn_best.n_neighbors})\nAccuracy: {knn_acc:.4f}', fontweight='bold')
plt.tight_layout()
plt.show()

In [ ]:
# Decision boundary visualization (PCA 2D)
pca_2d = PCA(n_components=2, random_state=RANDOM_STATE)
X_train_2d = pca_2d.fit_transform(X_train)
X_test_2d = pca_2d.transform(X_test)

explained = pca_2d.explained_variance_ratio_
print(f'PCA explained variance: PC1={explained[0]:.4f}, PC2={explained[1]:.4f}, Total={sum(explained):.4f}')

# Fit KNN on 2D data
knn_2d = KNeighborsClassifier(n_neighbors=knn_best.n_neighbors,
                               weights=knn_best.weights, metric=knn_best.metric)
knn_2d.fit(X_train_2d, y_train)

# Mesh grid
h = 0.3
x_min, x_max = X_train_2d[:, 0].min() - 1, X_train_2d[:, 0].max() + 1
y_min, y_max = X_train_2d[:, 1].min() - 1, X_train_2d[:, 1].max() + 1
xx, yy = np.meshgrid(np.arange(x_min, x_max, h), np.arange(y_min, y_max, h))
Z = knn_2d.predict(np.c_[xx.ravel(), yy.ravel()]).reshape(xx.shape)

fig, ax = plt.subplots(figsize=(10, 8))
cmap_bg = plt.cm.get_cmap('RdYlGn', 2)
ax.contourf(xx, yy, Z, alpha=0.25, cmap=cmap_bg)
ax.contour(xx, yy, Z, colors='gray', linewidths=0.5, alpha=0.5)

for i, cls in enumerate([0, 1]):
    mask = y_test == cls
    ax.scatter(X_test_2d[mask, 0], X_test_2d[mask, 1],
               c=CLASS_COLORS[i], label=CLASS_NAMES[cls],
               edgecolors='black', linewidth=0.5, s=30, alpha=0.75)

ax.set_xlabel(f'PC1 ({explained[0]*100:.1f}% variance)', fontsize=13)
ax.set_ylabel(f'PC2 ({explained[1]*100:.1f}% variance)', fontsize=13)
ax.set_title(f'KNN Decision Boundary (k={knn_best.n_neighbors}, PCA 2D)\n'
             f'2D Test Accuracy: {knn_2d.score(X_test_2d, y_test):.4f}',
             fontsize=14, fontweight='bold')
ax.legend(fontsize=11)
ax.grid(True, alpha=0.2)
plt.tight_layout()
plt.show()

---
## 11. Model 3: SVM Linear

Support Vector Machine with a linear kernel fits a maximum-margin hyperplane. For ternary features, the linear boundary effectively combines URL indicators into a phishing score.

**Key concept:** The margin is the distance between the decision boundary and the nearest training points (support vectors). Maximizing this margin leads to better generalization.

In [ ]:
# Hyperparameter tuning
print('SVM Linear -- Hyperparameter Tuning (GridSearchCV, 5-fold CV)')
print('=' * 60)

svm_lin_param_grid = {
    'C': [0.001, 0.01, 0.1, 1, 10],
}

svm_lin = SVC(kernel='linear', random_state=RANDOM_STATE)
svm_lin_grid = GridSearchCV(
    estimator=svm_lin,
    param_grid=svm_lin_param_grid,
    cv=5,
    scoring='f1_weighted',
    n_jobs=-1,
    refit=True,
)
svm_lin_grid.fit(X_train, y_train)

print(f'\nBest parameters: {svm_lin_grid.best_params_}')
print(f'Best CV F1 (weighted): {svm_lin_grid.best_score_:.4f}')

svm_lin_best = svm_lin_grid.best_estimator_

# All CV results
print(f'\n{"C":<10} {"Mean F1":<12} {"Std F1":<12}')
print('-' * 34)
cv_res = svm_lin_grid.cv_results_
for i in range(len(cv_res['params'])):
    c_val = cv_res['params'][i]['C']
    marker = ' <-- best' if cv_res['params'][i] == svm_lin_grid.best_params_ else ''
    print(f'{c_val:<10} {cv_res["mean_test_score"][i]:<12.4f} {cv_res["std_test_score"][i]:<12.4f}{marker}')

In [ ]:
# Evaluation
svm_lin_pred = svm_lin_best.predict(X_test)
svm_lin_acc = accuracy_score(y_test, svm_lin_pred)
svm_lin_prec = precision_score(y_test, svm_lin_pred, average='weighted')
svm_lin_rec = recall_score(y_test, svm_lin_pred, average='weighted')
svm_lin_f1 = f1_score(y_test, svm_lin_pred, average='weighted')

print('SVM Linear -- Test Set Results')
print('=' * 45)
print(f'Accuracy:  {svm_lin_acc:.4f}')
print(f'Precision: {svm_lin_prec:.4f}')
print(f'Recall:    {svm_lin_rec:.4f}')
print(f'F1-Score:  {svm_lin_f1:.4f}')
print(f'\nSupport vectors: {sum(svm_lin_best.n_support_)} / {X_train.shape[0]} '
      f'({sum(svm_lin_best.n_support_)/X_train.shape[0]*100:.1f}%)')
print(f'\n{classification_report(y_test, svm_lin_pred, target_names=CLASS_NAMES)}')

# Confusion matrix and decision boundary
svm_lin_cm = confusion_matrix(y_test, svm_lin_pred)

fig, axes = plt.subplots(1, 2, figsize=(16, 6))

# Confusion matrix
sns.heatmap(svm_lin_cm, annot=True, fmt='d', cmap='Blues',
            xticklabels=CLASS_NAMES, yticklabels=CLASS_NAMES,
            ax=axes[0], linewidths=0.5)
axes[0].set_xlabel('Predicted')
axes[0].set_ylabel('Actual')
axes[0].set_title(f'SVM Linear Confusion Matrix\nAccuracy: {svm_lin_acc:.4f}', fontweight='bold')

# Decision boundary (PCA 2D)
svm_2d_lin = SVC(kernel='linear', C=svm_lin_grid.best_params_['C'], random_state=RANDOM_STATE)
svm_2d_lin.fit(X_train_2d, y_train)
Z_lin = svm_2d_lin.predict(np.c_[xx.ravel(), yy.ravel()]).reshape(xx.shape)

axes[1].contourf(xx, yy, Z_lin, alpha=0.3, cmap=cmap_bg)
for i, cls in enumerate([0, 1]):
    mask = y_test == cls
    axes[1].scatter(X_test_2d[mask, 0], X_test_2d[mask, 1],
                    c=CLASS_COLORS[i], label=CLASS_NAMES[cls],
                    edgecolors='k', s=15, alpha=0.6)
axes[1].set_xlabel(f'PC1 ({explained[0]*100:.1f}%)')
axes[1].set_ylabel(f'PC2 ({explained[1]*100:.1f}%)')
axes[1].set_title(f'SVM Linear Decision Boundary (PCA 2D)\n'
                  f'C={svm_lin_grid.best_params_["C"]}', fontweight='bold')
axes[1].legend(fontsize=9)

plt.tight_layout()
plt.show()

---
## 12. Model 4: SVM RBF

SVM with the RBF (Radial Basis Function) kernel maps data into a higher-dimensional space where non-linear patterns become linearly separable.

**Key parameters:**
- **C** controls the trade-off between margin width and misclassification
- **gamma** controls how far the influence of each training example reaches (high gamma = tight fit around each point)

In [ ]:
# Hyperparameter tuning
print('SVM RBF -- Hyperparameter Tuning (GridSearchCV, 5-fold CV)')
print('=' * 55)

svm_rbf_param_grid = {
    'C': [0.1, 1, 10, 100],
    'gamma': ['scale', 'auto', 0.01, 0.1],
}

svm_rbf = SVC(kernel='rbf', random_state=RANDOM_STATE)
svm_rbf_grid = GridSearchCV(
    estimator=svm_rbf,
    param_grid=svm_rbf_param_grid,
    cv=5,
    scoring='f1_weighted',
    n_jobs=-1,
    refit=True,
)
svm_rbf_grid.fit(X_train, y_train)

print(f'\nBest parameters: {svm_rbf_grid.best_params_}')
print(f'Best CV F1 (weighted): {svm_rbf_grid.best_score_:.4f}')

svm_rbf_best = svm_rbf_grid.best_estimator_

In [ ]:
# Evaluation
svm_rbf_pred = svm_rbf_best.predict(X_test)
svm_rbf_acc = accuracy_score(y_test, svm_rbf_pred)
svm_rbf_prec = precision_score(y_test, svm_rbf_pred, average='weighted')
svm_rbf_rec = recall_score(y_test, svm_rbf_pred, average='weighted')
svm_rbf_f1 = f1_score(y_test, svm_rbf_pred, average='weighted')

print('SVM RBF -- Test Set Results')
print('=' * 45)
print(f'Accuracy:  {svm_rbf_acc:.4f}')
print(f'Precision: {svm_rbf_prec:.4f}')
print(f'Recall:    {svm_rbf_rec:.4f}')
print(f'F1-Score:  {svm_rbf_f1:.4f}')
print(f'\n{classification_report(y_test, svm_rbf_pred, target_names=CLASS_NAMES)}')

# Confusion matrix and decision boundary comparison
svm_rbf_cm = confusion_matrix(y_test, svm_rbf_pred)

fig, axes = plt.subplots(1, 3, figsize=(21, 6))

# Confusion matrix
sns.heatmap(svm_rbf_cm, annot=True, fmt='d', cmap='Blues',
            xticklabels=CLASS_NAMES, yticklabels=CLASS_NAMES,
            ax=axes[0], linewidths=0.5)
axes[0].set_xlabel('Predicted')
axes[0].set_ylabel('Actual')
axes[0].set_title(f'SVM RBF Confusion Matrix\nAccuracy: {svm_rbf_acc:.4f}', fontweight='bold')

# Decision boundary (PCA 2D)
svm_2d_rbf = SVC(kernel='rbf', C=svm_rbf_grid.best_params_['C'],
                 gamma=svm_rbf_grid.best_params_['gamma'], random_state=RANDOM_STATE)
svm_2d_rbf.fit(X_train_2d, y_train)
Z_rbf = svm_2d_rbf.predict(np.c_[xx.ravel(), yy.ravel()]).reshape(xx.shape)

axes[1].contourf(xx, yy, Z_rbf, alpha=0.3, cmap=cmap_bg)
for i, cls in enumerate([0, 1]):
    mask = y_test == cls
    axes[1].scatter(X_test_2d[mask, 0], X_test_2d[mask, 1],
                    c=CLASS_COLORS[i], label=CLASS_NAMES[cls],
                    edgecolors='k', s=15, alpha=0.6)
axes[1].set_xlabel(f'PC1 ({explained[0]*100:.1f}%)')
axes[1].set_ylabel(f'PC2 ({explained[1]*100:.1f}%)')
axes[1].set_title('SVM RBF Decision Boundary (PCA 2D)', fontweight='bold')
axes[1].legend(fontsize=9)

# Compare Linear vs RBF boundaries
axes[2].contourf(xx, yy, Z_lin, alpha=0.15, cmap=plt.cm.get_cmap('Blues', 2))
axes[2].contour(xx, yy, Z_lin, colors='blue', linewidths=2, linestyles='--')
axes[2].contour(xx, yy, Z_rbf, colors='red', linewidths=2, linestyles='-')
for i, cls in enumerate([0, 1]):
    mask = y_test == cls
    axes[2].scatter(X_test_2d[mask, 0], X_test_2d[mask, 1],
                    c=CLASS_COLORS[i], label=CLASS_NAMES[cls],
                    edgecolors='k', s=12, alpha=0.5)
axes[2].set_xlabel(f'PC1 ({explained[0]*100:.1f}%)')
axes[2].set_ylabel(f'PC2 ({explained[1]*100:.1f}%)')
axes[2].set_title('Linear (blue dashed) vs RBF (red solid)\nDecision Boundary Comparison', fontweight='bold')
axes[2].legend(fontsize=9)

plt.tight_layout()
plt.show()

print(f'\nComparison: SVM Linear Acc={svm_lin_acc:.4f} vs SVM RBF Acc={svm_rbf_acc:.4f}')
print(f'Improvement: {(svm_rbf_acc - svm_lin_acc)*100:+.2f} percentage points')

---
## 13. Model 5: Decision Tree

Decision Trees recursively partition the feature space using axis-aligned splits. They are highly interpretable -- the tree structure directly shows which features and thresholds drive classification decisions.

**Key parameters:**
- **max_depth** controls tree complexity (deeper = more complex, risk of overfitting)
- **min_samples_split** prevents splits on small subsets

In [ ]:
# Hyperparameter tuning
print('Decision Tree -- Hyperparameter Tuning (GridSearchCV, 5-fold CV)')
print('=' * 65)

dt_param_grid = {
    'max_depth': [3, 5, 7, 10, 15, 20, None],
    'min_samples_split': [2, 5, 10, 20],
    'min_samples_leaf': [1, 2, 5, 10],
    'criterion': ['gini', 'entropy'],
}

dt = DecisionTreeClassifier(random_state=RANDOM_STATE)
dt_grid = GridSearchCV(
    estimator=dt,
    param_grid=dt_param_grid,
    cv=5,
    scoring='accuracy',
    n_jobs=-1,
    refit=True,
)
dt_grid.fit(X_train, y_train)

print(f'\nBest parameters: {dt_grid.best_params_}')
print(f'Best CV accuracy: {dt_grid.best_score_:.4f}')

dt_best = dt_grid.best_estimator_

In [ ]:
# Evaluation
dt_pred = dt_best.predict(X_test)
dt_acc = accuracy_score(y_test, dt_pred)
dt_prec = precision_score(y_test, dt_pred, average='weighted')
dt_rec = recall_score(y_test, dt_pred, average='weighted')
dt_f1 = f1_score(y_test, dt_pred, average='weighted')

print('Decision Tree -- Test Set Results')
print('=' * 45)
print(f'Accuracy:  {dt_acc:.4f}')
print(f'Precision: {dt_prec:.4f}')
print(f'Recall:    {dt_rec:.4f}')
print(f'F1-Score:  {dt_f1:.4f}')
print(f'Tree depth: {dt_best.get_depth()}')
print(f'Number of leaves: {dt_best.get_n_leaves()}')
print(f'\n{classification_report(y_test, dt_pred, target_names=CLASS_NAMES)}')

# Confusion matrix
dt_cm = confusion_matrix(y_test, dt_pred)

fig, ax = plt.subplots(figsize=(7, 5.5))
sns.heatmap(dt_cm, annot=True, fmt='d', cmap='Blues',
            xticklabels=CLASS_NAMES, yticklabels=CLASS_NAMES,
            ax=ax, linewidths=0.5)
ax.set_xlabel('Predicted')
ax.set_ylabel('Actual')
ax.set_title(f'Decision Tree Confusion Matrix\nAccuracy: {dt_acc:.4f}', fontweight='bold')
plt.tight_layout()
plt.show()

In [ ]:
# Tree visualization (first 4 levels)
fig, ax = plt.subplots(figsize=(24, 10))
plot_tree(dt_best, max_depth=4, feature_names=feature_cols,
          class_names=CLASS_NAMES, filled=True, rounded=True,
          fontsize=8, ax=ax, proportion=True)
ax.set_title('Decision Tree Structure (Top 4 Levels)', fontsize=16, fontweight='bold')
plt.tight_layout()
plt.show()

In [ ]:
# Feature importance from decision tree
dt_importance = pd.DataFrame({
    'Feature': feature_cols,
    'Importance': dt_best.feature_importances_
}).sort_values('Importance', ascending=True)

fig, ax = plt.subplots(figsize=(10, 9))
ax.barh(dt_importance['Feature'], dt_importance['Importance'],
        color='#8e44ad', edgecolor='black', linewidth=0.5)
ax.set_xlabel('Feature Importance (Gini)')
ax.set_title('Decision Tree Feature Importance', fontsize=13, fontweight='bold')
ax.tick_params(axis='y', labelsize=9)
plt.tight_layout()
plt.show()

In [ ]:
# Overfitting analysis: depth vs accuracy
depths = list(range(1, 31))
train_accs_dt = []
test_accs_dt = []

for d in depths:
    dt_temp = DecisionTreeClassifier(max_depth=d, random_state=RANDOM_STATE)
    dt_temp.fit(X_train, y_train)
    train_accs_dt.append(accuracy_score(y_train, dt_temp.predict(X_train)))
    test_accs_dt.append(accuracy_score(y_test, dt_temp.predict(X_test)))

fig, ax = plt.subplots(figsize=(10, 6))
ax.plot(depths, train_accs_dt, 'o-', label='Training Accuracy', color='#2196F3', linewidth=2)
ax.plot(depths, test_accs_dt, 's-', label='Test Accuracy', color='#FF5722', linewidth=2)

best_depth = dt_best.get_depth()
ax.axvline(x=best_depth, color='green', linestyle='--', alpha=0.7, label=f'Best depth={best_depth}')

ax.set_xlabel('Max Depth', fontsize=13)
ax.set_ylabel('Accuracy', fontsize=13)
ax.set_title('Decision Tree: Depth vs Accuracy (Overfitting Analysis)',
             fontsize=14, fontweight='bold')
ax.legend(fontsize=11)
ax.grid(True, alpha=0.3)

# Shade overfitting region
overfit_start = next((d for d in depths if train_accs_dt[d-1] - test_accs_dt[d-1] > 0.05), None)
if overfit_start:
    ax.axvspan(overfit_start, max(depths), alpha=0.1, color='red', label='Overfitting region')

plt.tight_layout()
plt.show()

print(f'\nOverfitting analysis:')
print(f'  Best test accuracy at depth {depths[np.argmax(test_accs_dt)]}: {max(test_accs_dt):.4f}')
print(f'  Training accuracy at same depth: {train_accs_dt[np.argmax(test_accs_dt)]:.4f}')
print(f'  Gap: {train_accs_dt[np.argmax(test_accs_dt)] - max(test_accs_dt):.4f}')

---
## 14. Model 6: Multi-Layer Perceptron (MLP)

The MLP is a feedforward neural network with one or more hidden layers. It can learn complex, non-linear decision boundaries through backpropagation.

**Key parameters:**
- **hidden_layer_sizes** controls the network architecture
- **alpha** is the L2 regularization strength
- **learning_rate_init** affects convergence speed

In [ ]:
# Hyperparameter tuning
print('MLP -- Hyperparameter Tuning (GridSearchCV, 5-fold CV)')
print('=' * 55)

mlp_param_grid = {
    'hidden_layer_sizes': [(50,), (100,), (50, 50), (100, 50)],
    'alpha': [0.0001, 0.001, 0.01],
    'learning_rate_init': [0.001, 0.01],
}

mlp = MLPClassifier(max_iter=500, random_state=RANDOM_STATE, early_stopping=True,
                    validation_fraction=0.1)
mlp_grid = GridSearchCV(
    estimator=mlp,
    param_grid=mlp_param_grid,
    cv=5,
    scoring='accuracy',
    n_jobs=-1,
    refit=True,
)
mlp_grid.fit(X_train, y_train)

print(f'\nBest parameters: {mlp_grid.best_params_}')
print(f'Best CV accuracy: {mlp_grid.best_score_:.4f}')

mlp_best = mlp_grid.best_estimator_

In [ ]:
# Evaluation
mlp_pred = mlp_best.predict(X_test)
mlp_acc = accuracy_score(y_test, mlp_pred)
mlp_prec = precision_score(y_test, mlp_pred, average='weighted')
mlp_rec = recall_score(y_test, mlp_pred, average='weighted')
mlp_f1 = f1_score(y_test, mlp_pred, average='weighted')

print('MLP -- Test Set Results')
print('=' * 45)
print(f'Accuracy:  {mlp_acc:.4f}')
print(f'Precision: {mlp_prec:.4f}')
print(f'Recall:    {mlp_rec:.4f}')
print(f'F1-Score:  {mlp_f1:.4f}')
print(f'Number of iterations: {mlp_best.n_iter_}')
print(f'Architecture: {mlp_best.hidden_layer_sizes}')
print(f'\n{classification_report(y_test, mlp_pred, target_names=CLASS_NAMES)}')

# Confusion matrix and loss curve
mlp_cm = confusion_matrix(y_test, mlp_pred)

fig, axes = plt.subplots(1, 2, figsize=(16, 5.5))

# Confusion matrix
sns.heatmap(mlp_cm, annot=True, fmt='d', cmap='Blues',
            xticklabels=CLASS_NAMES, yticklabels=CLASS_NAMES,
            ax=axes[0], linewidths=0.5)
axes[0].set_xlabel('Predicted')
axes[0].set_ylabel('Actual')
axes[0].set_title(f'MLP Confusion Matrix\nAccuracy: {mlp_acc:.4f}', fontweight='bold')

# Loss curve
if hasattr(mlp_best, 'loss_curve_'):
    axes[1].plot(mlp_best.loss_curve_, color='#e74c3c', linewidth=2, label='Training Loss')
    if hasattr(mlp_best, 'validation_scores_') and mlp_best.validation_scores_ is not None:
        axes[1].plot(mlp_best.validation_scores_, color='#2ecc71', linewidth=2, label='Validation Score')
    axes[1].set_xlabel('Iteration')
    axes[1].set_ylabel('Loss / Score')
    axes[1].set_title('MLP Training Curve', fontweight='bold')
    axes[1].legend()
    axes[1].grid(True, alpha=0.3)

plt.tight_layout()
plt.show()

In [ ]:
# Architecture comparison: train different architectures and compare
architectures = {
    '(50,)': (50,),
    '(100,)': (100,),
    '(50,50)': (50, 50),
    '(100,50)': (100, 50),
    '(100,100)': (100, 100),
    '(100,50,25)': (100, 50, 25),
}

arch_results = []
for name, layers in architectures.items():
    mlp_temp = MLPClassifier(hidden_layer_sizes=layers, max_iter=500,
                            random_state=RANDOM_STATE, early_stopping=True)
    mlp_temp.fit(X_train, y_train)
    temp_acc = accuracy_score(y_test, mlp_temp.predict(X_test))
    temp_f1_val = f1_score(y_test, mlp_temp.predict(X_test), average='weighted')
    n_params = sum(w.size for w in mlp_temp.coefs_) + sum(b.size for b in mlp_temp.intercepts_)
    arch_results.append({'Architecture': name, 'Accuracy': temp_acc, 'F1': temp_f1_val, 'Parameters': n_params})

arch_df = pd.DataFrame(arch_results).sort_values('Accuracy', ascending=False)
print('Architecture Comparison:')
print(arch_df.to_string(index=False))

fig, ax = plt.subplots(figsize=(10, 5))
bars = ax.bar(arch_df['Architecture'], arch_df['Accuracy'],
              color=sns.color_palette('viridis', len(arch_df)), edgecolor='black')
for bar, val in zip(bars, arch_df['Accuracy']):
    ax.text(bar.get_x() + bar.get_width()/2, val + 0.001,
            f'{val:.4f}', ha='center', fontsize=9, fontweight='bold')
ax.set_xlabel('Architecture')
ax.set_ylabel('Test Accuracy')
ax.set_title('MLP Architecture Comparison', fontsize=14, fontweight='bold')
ax.tick_params(axis='x', rotation=30)
plt.tight_layout()
plt.show()

---
## 15. Model 7: K-Means + PCA (Unsupervised)

As an unsupervised baseline, we apply PCA for dimensionality reduction and K-Means clustering. We then evaluate cluster quality by mapping cluster labels to true class labels via majority vote.

**Key question:** Do phishing and legitimate websites naturally cluster into distinct groups in feature space?

In [ ]:
# PCA Analysis
print('PCA Analysis')
print('=' * 55)

pca_full = PCA(random_state=RANDOM_STATE)
X_train_pca_full = pca_full.fit_transform(X_train)

# Explained variance
cumulative_variance = np.cumsum(pca_full.explained_variance_ratio_)
print(f'\nComponents needed for 90% variance: {np.argmax(cumulative_variance >= 0.90) + 1}')
print(f'Components needed for 95% variance: {np.argmax(cumulative_variance >= 0.95) + 1}')
print(f'Components needed for 99% variance: {np.argmax(cumulative_variance >= 0.99) + 1}')

fig, axes = plt.subplots(1, 2, figsize=(16, 5))

# Individual explained variance
axes[0].bar(range(1, len(pca_full.explained_variance_ratio_) + 1),
            pca_full.explained_variance_ratio_, color='#3498db', edgecolor='black')
axes[0].set_xlabel('Principal Component')
axes[0].set_ylabel('Explained Variance Ratio')
axes[0].set_title('PCA Individual Explained Variance', fontweight='bold')

# Cumulative explained variance
axes[1].plot(range(1, len(cumulative_variance) + 1), cumulative_variance,
             'o-', color='#e74c3c', linewidth=2)
axes[1].axhline(y=0.90, color='gray', linestyle='--', alpha=0.7, label='90% variance')
axes[1].axhline(y=0.95, color='gray', linestyle=':', alpha=0.7, label='95% variance')
axes[1].set_xlabel('Number of Components')
axes[1].set_ylabel('Cumulative Explained Variance')
axes[1].set_title('PCA Cumulative Explained Variance', fontweight='bold')
axes[1].legend(fontsize=10)
axes[1].grid(True, alpha=0.3)

plt.suptitle('PCA Analysis -- Phishing Website Features', fontsize=14, fontweight='bold', y=1.02)
plt.tight_layout()
plt.show()

In [ ]:
# K-Means Clustering
print('K-Means Clustering (k=2)')
print('=' * 55)

kmeans = KMeans(n_clusters=2, random_state=RANDOM_STATE, n_init=10, max_iter=300)
kmeans.fit(X_train)

# Predict cluster labels on test set
km_cluster_labels = kmeans.predict(X_test)

# Majority-vote label mapping
label_map = {}
for c in np.unique(km_cluster_labels):
    mask = km_cluster_labels == c
    if mask.sum() > 0:
        label_map[c] = int(stats.mode(y_test[mask], keepdims=True).mode[0])

km_mapped_preds = np.array([label_map.get(c, -1) for c in km_cluster_labels])

print(f'\nCluster label mapping (majority vote):')
for cluster, true_label in label_map.items():
    print(f'  Cluster {cluster} -> {CLASS_NAMES[true_label]} ({true_label})')

km_acc = accuracy_score(y_test, km_mapped_preds)
km_prec = precision_score(y_test, km_mapped_preds, average='weighted', zero_division=0)
km_rec = recall_score(y_test, km_mapped_preds, average='weighted', zero_division=0)
km_f1 = f1_score(y_test, km_mapped_preds, average='weighted', zero_division=0)

print(f'\nK-Means Test Set Results (via majority vote):')
print(f'  Accuracy:  {km_acc:.4f}')
print(f'  Precision: {km_prec:.4f}')
print(f'  Recall:    {km_rec:.4f}')
print(f'  F1-Score:  {km_f1:.4f}')
print(f'  Inertia:   {kmeans.inertia_:.2f}')

In [ ]:
# Cluster visualization in PCA 2D space
X_test_pca2 = pca_2d.transform(X_test)
centroids_pca2 = pca_2d.transform(kmeans.cluster_centers_)

fig, axes = plt.subplots(1, 3, figsize=(21, 6))

# True labels
for i, cls in enumerate([0, 1]):
    mask = y_test == cls
    axes[0].scatter(X_test_pca2[mask, 0], X_test_pca2[mask, 1],
                    c=CLASS_COLORS[i], label=CLASS_NAMES[cls],
                    edgecolors='k', s=20, alpha=0.6)
axes[0].set_title('True Labels (PCA 2D)', fontweight='bold')
axes[0].legend(fontsize=10)
axes[0].set_xlabel(f'PC1 ({explained[0]*100:.1f}%)')
axes[0].set_ylabel(f'PC2 ({explained[1]*100:.1f}%)')

# K-Means cluster labels
cluster_colors = ['#9b59b6', '#1abc9c']
for c in [0, 1]:
    mask = km_cluster_labels == c
    axes[1].scatter(X_test_pca2[mask, 0], X_test_pca2[mask, 1],
                    c=cluster_colors[c], label=f'Cluster {c}',
                    edgecolors='k', s=20, alpha=0.6)
axes[1].scatter(centroids_pca2[:, 0], centroids_pca2[:, 1],
                c='yellow', marker='X', s=300, edgecolors='black', linewidth=2,
                zorder=10, label='Centroids')
axes[1].set_title('K-Means Cluster Labels (PCA 2D)', fontweight='bold')
axes[1].legend(fontsize=10)
axes[1].set_xlabel(f'PC1 ({explained[0]*100:.1f}%)')
axes[1].set_ylabel(f'PC2 ({explained[1]*100:.1f}%)')

# Mapped predictions vs true
correct = km_mapped_preds == y_test
axes[2].scatter(X_test_pca2[correct, 0], X_test_pca2[correct, 1],
                c='#2ecc71', label='Correct', s=20, alpha=0.5)
axes[2].scatter(X_test_pca2[~correct, 0], X_test_pca2[~correct, 1],
                c='#e74c3c', label='Incorrect', s=20, alpha=0.5)
axes[2].set_title(f'K-Means Classification (Acc: {km_acc:.4f})', fontweight='bold')
axes[2].legend(fontsize=10)
axes[2].set_xlabel(f'PC1 ({explained[0]*100:.1f}%)')
axes[2].set_ylabel(f'PC2 ({explained[1]*100:.1f}%)')

plt.suptitle('K-Means Clustering Analysis -- Phishing Website Detection',
             fontsize=14, fontweight='bold', y=1.02)
plt.tight_layout()
plt.show()

In [ ]:
# Elbow method to validate k=2
k_range = range(2, 11)
inertias = []
for k in k_range:
    km_temp = KMeans(n_clusters=k, random_state=RANDOM_STATE, n_init=10)
    km_temp.fit(X_train)
    inertias.append(km_temp.inertia_)

fig, ax = plt.subplots(figsize=(10, 5))
ax.plot(list(k_range), inertias, 'o-', color='#e74c3c', linewidth=2, markersize=8)
ax.scatter([2], [inertias[0]], color='green', s=200, zorder=5, edgecolors='black', linewidth=2, label='k=2 (chosen)')
ax.set_xlabel('Number of Clusters (k)', fontsize=13)
ax.set_ylabel('Inertia (Within-Cluster Sum of Squares)', fontsize=13)
ax.set_title('K-Means Elbow Method', fontsize=14, fontweight='bold')
ax.legend(fontsize=11)
ax.grid(True, alpha=0.3)
plt.tight_layout()
plt.show()

---
## 16. Final Comparison

We now compare all models on the same test set using multiple metrics: Accuracy, Precision, Recall, F1-Score, and ROC-AUC. We also perform McNemar's test for statistical significance between top models.

In [ ]:
# Collect all results
all_results = [
    {'Model': 'Logistic Regression', 'Accuracy': lr_acc, 'Precision': lr_prec,
     'Recall': lr_rec, 'F1': lr_f1, 'pred': lr_pred},
    {'Model': 'KNN', 'Accuracy': knn_acc, 'Precision': knn_prec,
     'Recall': knn_rec, 'F1': knn_f1, 'pred': knn_pred},
    {'Model': 'SVM Linear', 'Accuracy': svm_lin_acc, 'Precision': svm_lin_prec,
     'Recall': svm_lin_rec, 'F1': svm_lin_f1, 'pred': svm_lin_pred},
    {'Model': 'SVM RBF', 'Accuracy': svm_rbf_acc, 'Precision': svm_rbf_prec,
     'Recall': svm_rbf_rec, 'F1': svm_rbf_f1, 'pred': svm_rbf_pred},
    {'Model': 'Decision Tree', 'Accuracy': dt_acc, 'Precision': dt_prec,
     'Recall': dt_rec, 'F1': dt_f1, 'pred': dt_pred},
    {'Model': 'MLP', 'Accuracy': mlp_acc, 'Precision': mlp_prec,
     'Recall': mlp_rec, 'F1': mlp_f1, 'pred': mlp_pred},
    {'Model': 'K-Means*', 'Accuracy': km_acc, 'Precision': km_prec,
     'Recall': km_rec, 'F1': km_f1, 'pred': km_mapped_preds},
]

# Create comparison DataFrame
comparison_df = pd.DataFrame([
    {k: v for k, v in r.items() if k != 'pred'} for r in all_results
]).sort_values('Accuracy', ascending=False).reset_index(drop=True)

# Compute ROC-AUC for models with probability support
roc_auc_scores = {}
for r in all_results:
    name = r['Model']
    if name == 'Logistic Regression':
        roc_auc_scores[name] = roc_auc_score(y_test, lr_best.predict_proba(X_test)[:, 1])
    elif name == 'KNN':
        roc_auc_scores[name] = roc_auc_score(y_test, knn_best.predict_proba(X_test)[:, 1])
    elif name == 'MLP':
        roc_auc_scores[name] = roc_auc_score(y_test, mlp_best.predict_proba(X_test)[:, 1])
    elif name == 'Decision Tree':
        roc_auc_scores[name] = roc_auc_score(y_test, dt_best.predict_proba(X_test)[:, 1])
    elif name == 'SVM Linear':
        roc_auc_scores[name] = roc_auc_score(y_test, svm_lin_best.decision_function(X_test))
    elif name == 'SVM RBF':
        roc_auc_scores[name] = roc_auc_score(y_test, svm_rbf_best.decision_function(X_test))
    else:
        roc_auc_scores[name] = None

comparison_df['ROC-AUC'] = comparison_df['Model'].map(roc_auc_scores)

print('=' * 80)
print('FINAL MODEL COMPARISON -- Phishing Website Detection')
print('=' * 80)
print(comparison_df.to_string(index=False, float_format='{:.4f}'.format))
print('\n* K-Means uses majority-vote label mapping (unsupervised baseline)')

In [ ]:
# Visualization 1: Accuracy bar chart
fig, ax = plt.subplots(figsize=(10, 6))
colors_bar = sns.color_palette('viridis', n_colors=len(comparison_df))
bars = ax.barh(comparison_df['Model'], comparison_df['Accuracy'],
               color=colors_bar, edgecolor='white')
for bar, val in zip(bars, comparison_df['Accuracy']):
    ax.text(val + 0.003, bar.get_y() + bar.get_height() / 2,
            f'{val:.4f}', va='center', fontsize=10, fontweight='bold')
ax.set_xlabel('Accuracy', fontsize=12)
ax.set_title('Model Comparison -- Accuracy on Test Set\n(Phishing Website Detection)',
             fontsize=14, fontweight='bold')
ax.set_xlim(0, min(1.0, comparison_df['Accuracy'].max() + 0.08))
ax.invert_yaxis()
plt.tight_layout()
plt.show()

In [ ]:
# Visualization 2: Grouped bar chart (Precision / Recall / F1)
models = comparison_df['Model'].tolist()
metrics = ['Precision', 'Recall', 'F1']
x = np.arange(len(models))
width = 0.25

fig, ax = plt.subplots(figsize=(12, 6))
palette = ['#4C72B0', '#55A868', '#C44E52']
for i, (metric, color) in enumerate(zip(metrics, palette)):
    vals = comparison_df[metric].tolist()
    ax.bar(x + i * width, vals, width, label=metric, color=color)

ax.set_xticks(x + width)
ax.set_xticklabels(models, rotation=30, ha='right', fontsize=9)
ax.set_ylabel('Score', fontsize=12)
ax.set_title('Precision / Recall / F1 by Model -- Phishing Detection',
             fontsize=14, fontweight='bold')
ax.legend(fontsize=10)
ax.set_ylim(0, 1.05)
plt.tight_layout()
plt.show()

In [ ]:
# Visualization 3: Radar / Spider chart
radar_metrics = ['Accuracy', 'Precision', 'Recall', 'F1']
radar_labels = ['Accuracy', 'Precision', 'Recall', 'F1']
num_vars = len(radar_metrics)
angles = np.linspace(0, 2 * np.pi, num_vars, endpoint=False).tolist()
angles += angles[:1]

fig, ax = plt.subplots(figsize=(8, 8), subplot_kw=dict(polar=True))
palette_radar = sns.color_palette('husl', n_colors=len(comparison_df))

for idx, row in comparison_df.iterrows():
    values = [row[m] for m in radar_metrics]
    values += values[:1]
    ax.plot(angles, values, 'o-', linewidth=2, label=row['Model'], color=palette_radar[idx])
    ax.fill(angles, values, alpha=0.08, color=palette_radar[idx])

ax.set_thetagrids(np.degrees(angles[:-1]), radar_labels, fontsize=11)
ax.set_ylim(0, 1.0)
ax.set_title('Multi-Metric Radar Comparison -- Phishing Detection',
             fontsize=14, fontweight='bold', y=1.08)
ax.legend(loc='upper right', bbox_to_anchor=(1.35, 1.12), fontsize=8)
plt.tight_layout()
plt.show()

In [ ]:
# Visualization 4: Side-by-side confusion matrices
all_preds = {r['Model']: r['pred'] for r in all_results}
n_models = len(all_preds)
cols_cm = 3
rows_cm = int(np.ceil(n_models / cols_cm))

fig, axes = plt.subplots(rows_cm, cols_cm, figsize=(6 * cols_cm, 5 * rows_cm))
if rows_cm == 1:
    axes = axes[np.newaxis, :]

for idx, (name, pred) in enumerate(all_preds.items()):
    r, c = divmod(idx, cols_cm)
    cm = confusion_matrix(y_test, pred)
    sns.heatmap(cm, annot=True, fmt='d', cmap='Blues',
                xticklabels=CLASS_NAMES, yticklabels=CLASS_NAMES,
                ax=axes[r][c], cbar=False)
    acc_val = accuracy_score(y_test, pred)
    axes[r][c].set_title(f'{name}\nAcc: {acc_val:.4f}', fontsize=11, fontweight='bold')
    axes[r][c].set_xlabel('Predicted')
    axes[r][c].set_ylabel('Actual')

# Hide empty subplots
for idx in range(n_models, rows_cm * cols_cm):
    r, c = divmod(idx, cols_cm)
    axes[r][c].axis('off')

fig.suptitle('Confusion Matrices -- All Models (Phishing Detection)',
             fontsize=15, fontweight='bold', y=1.01)
plt.tight_layout()
plt.show()

In [ ]:
# McNemar's significance test between top models
print('=' * 70)
print('Statistical Significance -- McNemar\'s Test (Top Model Pairs)')
print('=' * 70)

def mcnemar_test(y_true, pred_a, pred_b):
    """McNemar's test with continuity correction."""
    correct_a = (pred_a == y_true)
    correct_b = (pred_b == y_true)
    b01 = np.sum(~correct_a & correct_b)  # A wrong, B right
    b10 = np.sum(correct_a & ~correct_b)   # A right, B wrong
    if b01 + b10 == 0:
        return 0.0, 1.0
    chi2 = (abs(b01 - b10) - 1) ** 2 / (b01 + b10)
    p_value = 1 - stats.chi2.cdf(chi2, df=1)
    return chi2, p_value

# Rank models by accuracy (exclude K-Means)
supervised_results = [r for r in all_results if '*' not in r['Model']]
supervised_results.sort(key=lambda x: x['Accuracy'], reverse=True)

print(f'\n{"Model A":<25} {"Model B":<25} {"Chi2":>10} {"p-value":>12} {"Significant?"}')
print('-' * 85)

tested = set()
for i in range(min(4, len(supervised_results))):
    for j in range(i + 1, min(4, len(supervised_results))):
        a = supervised_results[i]
        b = supervised_results[j]
        key = tuple(sorted([a['Model'], b['Model']]))
        if key in tested:
            continue
        tested.add(key)
        chi2_val, pval = mcnemar_test(y_test, a['pred'], b['pred'])
        sig = 'YES (p < 0.05)' if pval < 0.05 else 'NO  (p >= 0.05)'
        print(f'{a["Model"]:<25} {b["Model"]:<25} {chi2_val:>10.3f} {pval:>12.4f} {sig}')

In [ ]:
# Final ranking table
print('\n' + '=' * 80)
print('FINAL RANKING (by Accuracy, descending)')
print('=' * 80)

for i, row in comparison_df.iterrows():
    auc_str = f'  AUC={row["ROC-AUC"]:.4f}' if pd.notna(row.get('ROC-AUC')) else ''
    print(f'  #{i+1}  {row["Model"]:25s}  Accuracy={row["Accuracy"]:.4f}  '
          f'F1={row["F1"]:.4f}{auc_str}')

best = comparison_df.iloc[0]
print(f'\n  BEST MODEL: {best["Model"]} (Accuracy={best["Accuracy"]:.4f}, F1={best["F1"]:.4f})')
print(f'\n  * K-Means uses majority-vote mapping and is included as an unsupervised baseline.')
print(f'    It should not be directly compared to supervised models.')

---
## 17. Conclusion

### Summary of Findings

1. **Dataset characteristics:** The UCI Phishing Websites dataset contains 11,055 samples with 30 ternary features ({-1, 0, 1}) and near-perfect class balance (44.3% Phishing vs 55.7% Legitimate). No missing values or significant preprocessing challenges.

2. **Key features:** `SSLfinal_State`, `URL_of_Anchor`, `web_traffic`, and `having_Sub_Domain` consistently emerged as the most important features across correlation analysis, mutual information, and random forest importance.

3. **Model performance:** All supervised models achieve strong accuracy on this dataset. The ternary encoding of features means that even linear models (Logistic Regression, Linear SVM) perform well, suggesting the phishing signal is substantially linearly separable.

4. **Unsupervised baseline:** K-Means clustering shows that phishing and legitimate websites do form partially separable clusters in feature space, though the separation is imperfect -- confirming that supervised models are needed for reliable detection.

5. **Statistical significance:** McNemar's tests reveal whether performance differences between top models are statistically significant at the p < 0.05 level.

### Recommendations

- For **deployment in production**, choose the model with the best trade-off between accuracy and interpretability. Logistic Regression offers feature coefficients that explain predictions, while SVM RBF and MLP may offer higher accuracy.
- For **phishing detection specifically**, recall on the Phishing class is critical -- missing a phishing site (false negative) is more dangerous than blocking a legitimate site (false positive).
- The ternary feature encoding limits the expressiveness of the data. Future work could incorporate continuous features (URL length in characters, domain age in days) for potentially better discrimination.